In [ ]:
!pip install -q earthengine-api pymannkendall

import ee, json
ee.Authenticate()
ee.Initialize(project='microclimas-cochabamba')

RUTA_GEOJSON = '/content/distritos_indicadores_ambientales_2024.geojson'
gj = json.load(open(RUTA_GEOJSON, encoding='utf-8'))
print('Distritos en el archivo:', len(gj['features']))
print(gj['features'][0]['properties'])

Distritos en el archivo: 15
{'codigo': '030101-distrito-1', 'municipio_ine': '030101', 'codigo_distrito': '1', 'nombre': 'Distrito 1', 'fuente': 'ciudatoslab', 'latitud': -17.367893, 'longitud': -66.127688, 'LST_C_mean': 23.7947705472169, 'LST_C_median': 23.835269437250453, 'LST_C_min': 19.422973054399932, 'LST_C_max': 27.614864272489687, 'LST_C_stdDev': 1.2723108618904702, 'NDVI_mean': 0.19219278, 'NDVI_median': 0.16745448, 'NDVI_min': 0.042864934, 'NDVI_max': 0.69365132, 'NDVI_stdDev': 0.088521063, 'pixeles_validos': 570, 'correlacion_pearson': -0.4983170798988722, 'valor_p': 4.151235248465838e-37, 'valor_p_FDR': 1.5567132181746895e-36, 'significativo_FDR_05': True}


In [ ]:
import pandas as pd, numpy as np, pymannkendall as mk, time
from scipy import stats

CAMPO_ID, CAMPO_NOMBRE = 'codigo_distrito', 'nombre'
AÑO_INI, AÑO_FIN = 2001, 2025

distritos = ee.FeatureCollection(gj).select([CAMPO_ID, CAMPO_NOMBRE])
lst_col = ee.ImageCollection('MODIS/061/MOD11A2').select('LST_Day_1km')
ndvi_col = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')
vacia = ee.Image.constant(0).updateMask(ee.Image.constant(0))

def media_mes(col, ini, fin):
    c = col.filterDate(ini, fin)
    return ee.Image(ee.Algorithms.If(c.size().gt(0), c.mean(), vacia))

def imagen_anual(y):
    bandas = []
    for m in range(1, 13):
        ini = ee.Date.fromYMD(y, m, 1)
        fin = ini.advance(1, 'month')
        bandas.append(media_mes(lst_col, ini, fin).multiply(0.02).subtract(273.15).rename(f'lst_{m:02d}'))
        bandas.append(media_mes(ndvi_col, ini, fin).multiply(0.0001).rename(f'ndvi_{m:02d}'))
    return ee.Image.cat(bandas)

filas = []
for y in range(AÑO_INI, AÑO_FIN + 1):
    for intento in range(6):
        try:
            fc = (imagen_anual(y)
                  .reduceRegions(collection=distritos, reducer=ee.Reducer.mean(), scale=500, tileScale=4)
                  .map(lambda f: f.setGeometry(None)))
            feats = fc.getInfo()['features']
            break
        except Exception as e:
            espera = 20 * (intento + 1)
            print(f'{y}: Earth Engine pidió esperar, reintento en {espera} s')
            time.sleep(espera)
    else:
        raise RuntimeError(f'No se pudo procesar {y}')
    for f in feats:
        p = f['properties']
        for m in range(1, 13):
            filas.append({CAMPO_ID: p[CAMPO_ID], CAMPO_NOMBRE: p[CAMPO_NOMBRE],
                          'fecha': f'{y}-{m:02d}',
                          'lst': p.get(f'lst_{m:02d}'), 'ndvi': p.get(f'ndvi_{m:02d}')})
    print('Listo', y)
    time.sleep(2)

df = pd.DataFrame(filas)
df['fecha'] = pd.to_datetime(df['fecha'])
df.to_csv('series_mensuales_distritos.csv', index=False)
print(df.head(), '\nFilas:', len(df), '(deberían ser 15 × 300 = 4500)')

Listo 2001
Listo 2002
Listo 2003
Listo 2004
Listo 2005
Listo 2006
Listo 2007
Listo 2008
Listo 2009
Listo 2010
Listo 2011
Listo 2012
Listo 2013
Listo 2014
Listo 2015
Listo 2016
Listo 2017
Listo 2018
Listo 2019
Listo 2020
Listo 2021
Listo 2022
Listo 2023
Listo 2024
Listo 2025
  codigo_distrito      nombre      fecha        lst      ndvi
0               1  Distrito 1 2001-01-01  25.942136  0.347452
1               1  Distrito 1 2001-02-01  29.048384  0.375362
2               1  Distrito 1 2001-03-01  23.465568  0.404929
3               1  Distrito 1 2001-04-01  27.365956  0.332839
4               1  Distrito 1 2001-05-01  25.142304  0.277863 
Filas: 4500 (deberían ser 15 × 300 = 4500)


In [ ]:
def analizar(serie):
    s = serie.set_index('fecha').asfreq('MS')
    v = s.iloc[:, 0]
    # Mann-Kendall estacional: compara el mismo mes entre años (quita el ciclo estacional)
    r = mk.seasonal_test(v.values, period=12)
    # Regresión lineal sobre anomalías (valor - promedio de ese mes)
    anom = v - v.groupby(v.index.month).transform('mean')
    ok = anom.dropna()
    t = (ok.index.year + (ok.index.month - 1) / 12).values
    lr = stats.linregress(t, ok.values)
    tcrit = stats.t.ppf(0.975, len(ok) - 2)
    return {
        'sen_pendiente_anual': round(float(r.slope), 4),
        'mk_p': round(float(r.p), 4),
        'mk_tendencia': r.trend,  # increasing / decreasing / no trend
        'significativa': bool(r.p < 0.05),
        'lineal_pendiente_anual': round(float(lr.slope), 4),
        'lineal_p': round(float(lr.pvalue), 4),
        'ic95': [round(float(lr.slope - tcrit * lr.stderr), 4),
                 round(float(lr.slope + tcrit * lr.stderr), 4)],
        'meses_validos': int(v.notna().sum()),
    }

salida = {'meta': {
    'fuente_lst': 'MODIS MOD11A2 v061 (LST diurna, 1 km)',
    'fuente_ndvi': 'MODIS MOD13Q1 v061 (NDVI, 250 m)',
    'periodo': f'{AÑO_INI}-01 a {AÑO_FIN}-12', 'resolucion': 'mensual',
    'metodos': 'Mann-Kendall estacional + pendiente de Sen; regresión lineal sobre anomalías con IC 95 %',
    'unidades': {'lst': '°C/año', 'ndvi': 'índice/año'}}, 'distritos': []}

for did, g in df.groupby(CAMPO_ID):
    g = g.sort_values('fecha')
    salida['distritos'].append({
        'id': str(did), 'nombre': g[CAMPO_NOMBRE].iloc[0],
        'serie': [{'fecha': f.strftime('%Y-%m'),
                   'lst': None if pd.isna(a) else round(a, 2),
                   'ndvi': None if pd.isna(b) else round(b, 4)}
                  for f, a, b in zip(g['fecha'], g['lst'], g['ndvi'])],
        'tendencias': {'lst': analizar(g[['fecha', 'lst']]),
                       'ndvi': analizar(g[['fecha', 'ndvi']])}})

json.dump(salida, open('tendencias_distritos.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(pd.DataFrame([{ 'distrito': d['nombre'], **{f'lst_{k}': v for k, v in d['tendencias']['lst'].items()} }
                    for d in salida['distritos']])[['distrito','lst_sen_pendiente_anual','lst_mk_p','lst_significativa']])

       distrito  lst_sen_pendiente_anual  lst_mk_p  lst_significativa
0    Distrito 1                  -0.0705    0.0000               True
1   Distrito 10                  -0.0825    0.0000               True
2   Distrito 11                  -0.0697    0.0000               True
3   Distrito 12                  -0.0713    0.0000               True
4   Distrito 13                  -0.0564    0.0001               True
5   Distrito 14                  -0.0974    0.0000               True
6   Distrito 15                  -0.0762    0.0000               True
7    Distrito 2                  -0.0693    0.0000               True
8    Distrito 3                  -0.0488    0.0001               True
9    Distrito 4                  -0.0750    0.0000               True
10   Distrito 5                  -0.0672    0.0000               True
11   Distrito 6                  -0.0647    0.0000               True
12   Distrito 7                  -0.0807    0.0000               True
13   Distrito 8     

In [ ]:
def tabla(d, col):
    out = []
    for did, g in d.groupby(CAMPO_ID):
        r = analizar(g.sort_values('fecha')[['fecha', col]])
        out.append({'distrito': g[CAMPO_NOMBRE].iloc[0], 'pendiente_anual': r['sen_pendiente_anual'],
                    'p': r['mk_p'], 'significativa': r['significativa']})
    return pd.DataFrame(out).sort_values('pendiente_anual', ascending=False)

# Prueba 1: solo 2001-2019, antes de que la órbita de Terra se corriera
print('=== LST 2001-2019 ===')
print(tabla(df[df['fecha'].dt.year <= 2019], 'lst'))

# Prueba 2: cada distrito comparado con el promedio del valle en el mismo mes.
# Esto cancela el efecto del satélite y muestra qué distritos se calientan MÁS o MENOS que el resto
df['lst_rel'] = df['lst'] - df.groupby('fecha')['lst'].transform('mean')
print('\n=== LST relativa al promedio del valle, 2001-2025 ===')
print(tabla(df, 'lst_rel'))

# Y de paso, la vegetación
print('\n=== NDVI 2001-2025 ===')
print(tabla(df, 'ndvi'))

=== LST 2001-2019 ===
       distrito  pendiente_anual       p  significativa
14   Distrito 9           0.0226  0.2217          False
8    Distrito 3           0.0216  0.1665          False
4   Distrito 13           0.0132  0.3077          False
10   Distrito 5           0.0117  0.4610          False
2   Distrito 11           0.0091  0.6065          False
3   Distrito 12           0.0057  0.6207          False
0    Distrito 1           0.0023  0.8163          False
11   Distrito 6           0.0019  0.9115          False
1   Distrito 10          -0.0029  0.8478          False
7    Distrito 2          -0.0030  0.9436          False
6   Distrito 15          -0.0045  0.7696          False
9    Distrito 4          -0.0079  0.6207          False
12   Distrito 7          -0.0268  0.0806          False
13   Distrito 8          -0.0295  0.0917          False
5   Distrito 14          -0.0362  0.0348           True

=== LST relativa al promedio del valle, 2001-2025 ===
       distrito  pendiente_

In [ ]:
AÑO_CORTE_LST = 2019
salida = {'meta': {
    'fuente_lst': 'MODIS MOD11A2 v061 (LST diurna, 1 km, satélite Terra)',
    'fuente_ndvi': 'MODIS MOD13Q1 v061 (NDVI, 250 m)',
    'periodo_lst': f'{AÑO_INI}-01 a {AÑO_CORTE_LST}-12',
    'periodo_ndvi': f'{AÑO_INI}-01 a {AÑO_FIN}-12',
    'resolucion': 'mensual',
    'metodos': 'Mann-Kendall estacional + pendiente de Sen; regresión lineal sobre anomalías con IC 95 %',
    'unidades': {'lst': '°C/año', 'ndvi': 'índice/año'},
    'nota_lst': 'Desde 2020 la órbita del satélite Terra se desplazó y pasa más temprano, lo que enfría artificialmente la temperatura medida. Por eso la tendencia de LST se calcula solo hasta 2019.'},
    'distritos': []}

for did, g in df.groupby(CAMPO_ID):
    g = g.sort_values('fecha')
    g_lst = g[g['fecha'].dt.year <= AÑO_CORTE_LST]
    salida['distritos'].append({
        'id': str(did), 'nombre': g[CAMPO_NOMBRE].iloc[0],
        'serie': [{'fecha': f.strftime('%Y-%m'),
                   'lst': None if (pd.isna(a) or f.year > AÑO_CORTE_LST) else round(a, 2),
                   'ndvi': None if pd.isna(b) else round(b, 4)}
                  for f, a, b in zip(g['fecha'], g['lst'], g['ndvi'])],
        'tendencias': {'lst': analizar(g_lst[['fecha', 'lst']]),
                       'ndvi': analizar(g[['fecha', 'ndvi']])}})

json.dump(salida, open('tendencias_distritos.json', 'w', encoding='utf-8'), ensure_ascii=False)
print('JSON listo con', len(salida['distritos']), 'distritos')

JSON listo con 15 distritos
